# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.0 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    

In [34]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import optuna.visualization.matplotlib as optuna_mpl
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import make_scorer,mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from scipy.stats import randint
import joblib  # For saving model
import os  # For mkdir
from scipy.stats import randint
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# Add any other imports you need here

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [ ]:
# train_df = pd.read_csv('Dataset/train.csv')
# test_df = pd.read_csv('Dataset/test.csv')
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.


# TODO: Separate predictors (X) and target (y: 'price')
# Identify categorical columns (adjust if needed based on train_df.columns)
# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('screenshots', exist_ok=True)

# Categorical columns
categorical_cols = ['location', 'city', 'Status', 'property_type']

# Create safe encoders
label_encoders = {}
train_encodings = {}  # Store train categories

for col in categorical_cols:
    print(f"Processing {col}...")

    # Get ALL unique values from BOTH train AND test
    all_categories = pd.concat([train_df[col], test_df[col]], axis=0).unique()
    all_categories = sorted(set(all_categories))  # Remove duplicates, sort

    le = LabelEncoder()
    le.fit(all_categories)  # Fit on combined unique values

    # Transform both
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

    label_encoders[col] = le
    train_encodings[col] = {cat: idx for cat, idx in zip(le.classes_, le.transform(le.classes_))}

# Save encoders AND train mappings for Streamlit
import pickle
with open('models/encoders.pkl', 'wb') as f:
    pickle.dump({
        'encoders': label_encoders,
        'train_categories': train_encodings  # Only train-seen for UI
    }, f)

print("Safe encoding complete! Encoders saved.")

# Separate features (ALL columns except price)
feature_cols = train_df.columns.drop('price').tolist()
X_train = train_df[feature_cols]
y_train = train_df['price']
X_test = test_df[feature_cols]
y_test = test_df['price']

print("X_train shape:", X_train.shape)
print("Features:", feature_cols)

Processing location...
Processing city...
Processing Status...
Processing property_type...
Safe encoding complete! Encoders saved.
X_train shape: (11128, 14)
Features: ['location', 'city', 'latitude', 'longitude', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Status', 'Size_ft²', 'BHK', 'rooms_num', 'property_type', 'verification_days']


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

GRID SEARCH CODE

In [ ]:
import trackio
import numpy as np

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

print("Starting Grid Search (60 combinations)...")
start_time = time.time()

param_grid = {
    'n_estimators': [50, 100, 150, 200],   # 4
    'max_depth': [10, 15, 20, 25, 30],     # 5
    'min_samples_split': [2, 5, 8]         # 3  -> 4*5*3 = 60
}
n_combinations = (
    len(param_grid['n_estimators']) *
    len(param_grid['max_depth']) *
    len(param_grid['min_samples_split'])
)

trackio.init(
    project="hyperparameter_comparison",
    name="grid_search",
    config={"method": "GridSearch", "cv": 5, "n_trials": n_combinations}
)

grid_search = GridSearchCV(
    rf, param_grid,
    cv=5,
    scoring=mae_scorer,
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

cv_means = -grid_search.cv_results_['mean_test_score']  # positive MAE
best_so_far = np.minimum.accumulate(cv_means)

print(f"\nTotal planned trials: {n_combinations}")
for i, (curr_mae, best_mae) in enumerate(zip(cv_means, best_so_far), start=1):
    print(f"Trial {i}/{n_combinations} | CV MAE: {curr_mae:.4f} | "
          f"Best so far: {best_mae:.4f}")
    trackio.log({
        "method": "GridSearch",
        "trial": i,
        "cv_mae": float(curr_mae),
        "best_cv_mae_so_far": float(best_mae)
    })

grid_time = time.time() - start_time

print(f"\n Grid Search completed: {n_combinations} trials in {grid_time:.2f}s")
print(f"Best CV MAE overall: {-grid_search.best_score_:.4f}")

trackio.log({
    "best_cv_mae": float(-grid_search.best_score_),
    "time_taken": float(grid_time),
    "n_iterations": int(n_combinations)
})
trackio.finish()

Starting Grid Search (60 combinations)...
* Trackio project initialized: hyperparameter_comparison
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* Created new run: grid_search
* Running on public URL: https://ba2204451aa10d97ca.gradio.live



Total planned trials: 60
Trial 1/60 | CV MAE: 15217.7313 | Best so far: 15217.7313
Trial 2/60 | CV MAE: 15227.3902 | Best so far: 15217.7313
Trial 3/60 | CV MAE: 15226.9907 | Best so far: 15217.7313
Trial 4/60 | CV MAE: 15206.3867 | Best so far: 15206.3867
Trial 5/60 | CV MAE: 15311.4974 | Best so far: 15206.3867
Trial 6/60 | CV MAE: 15333.2704 | Best so far: 15206.3867
Trial 7/60 | CV MAE: 15324.0491 | Best so far: 15206.3867
Trial 8/60 | CV MAE: 15309.4809 | Best so far: 15206.3867
Trial 9/60 | CV MAE: 15415.4889 | Best so far: 15206.3867
Trial 10/60 | CV MAE: 15435.4863 | Best so far: 15206.3867
Trial 11/60 | CV MAE: 15421.9307 | Best so far: 15206.3867
Trial 12/60 | CV MAE: 15415.0255 | Best so far: 15206.3867
Trial 13/60 | CV MAE: 13709.6048 | Best so far: 13709.6048
Trial 14/60 | CV MAE: 13694.3149 | Best so far: 13694.3149
Trial 15/60 | CV MAE: 13667.6048 | Best so far: 13667.6048
Trial 16/60 | CV MAE: 13677.9885 | Best so far: 13667.6048
Trial 17/60 | CV MAE: 13908.3357 | Best

In [ ]:
trackio.show()

RANDOMIZED SEARCH

In [9]:
# RANDOMIZED SEARCH – with per-trial logging like GridSearch
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

N_RANDOM_TRIALS = 60  # final budget

print(f"\n  Starting Random Search ({N_RANDOM_TRIALS} trials)...")
start_time = time.time()

trackio.init(
    project="hyperparameter_comparison",
    name="random_search",
    config={"method": "RandomSearch", "cv": 5, "n_trials": N_RANDOM_TRIALS}
)

# Full-range search space
param_random = {
    "n_estimators": randint(50, 201),    # 50–200
    "max_depth": randint(10, 31),        # 10–30
    "min_samples_split": randint(2, 11)  # 2–10
}

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_random,
    n_iter=N_RANDOM_TRIALS,
    cv=5,
    scoring=mae_scorer,   # negative MAE
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

random_time = time.time() - start_time

# Per-trial scores from cv_results_
cv_means = -random_search.cv_results_["mean_test_score"]  # positive MAE
best_so_far = np.minimum.accumulate(cv_means)

for i, (curr_mae, best_mae) in enumerate(zip(cv_means, best_so_far), start=1):
    trackio.log({
        "trial": int(i),
        "cv_mae": float(curr_mae),
        "best_cv_mae_so_far": float(best_mae)
    })

# Final summary logging
trackio.log({
    "best_cv_mae": float(-random_search.best_score_),  # convert from negative MAE
    "time_taken": float(random_time),
    "n_iterations": int(N_RANDOM_TRIALS),
    "best_params": random_search.best_params_
})
trackio.finish()

print(f"   Random Search: {N_RANDOM_TRIALS} trials, {random_time:.2f}s")
print(f"   Best CV MAE: {-random_search.best_score_:.4f}")
print(f"   Best params: {random_search.best_params_}")



  Starting Random Search (60 trials)...
* Trackio project initialized: hyperparameter_comparison
* Trackio metrics logged to: /root/.cache/huggingface/trackio
* Created new run: random_search
* Running on public URL: https://e1790d4f087c0a29c2.gradio.live


Fitting 5 folds for each of 60 candidates, totalling 300 fits
* Run finished. Uploading logs to Trackio (please wait...)
   Random Search: 60 trials, 1726.48s
   Best CV MAE: 13309.7826
   Best params: {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}


In [ ]:
trackio.show()

Bayesian optimization

In [10]:
# BAYESIAN OPTIMIZATION (OPTUNA) – with per-trial logging like GridSearch  final code
import optuna
from sklearn.model_selection import cross_val_score
import numpy as np

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

N_BAYES_TRIALS = 60 # final budget

trackio.init(
    project="hyperparameter_comparison",
    name="bayesian_optuna",
    config={"method": "BayesianOptuna", "cv": 5, "n_trials": N_BAYES_TRIALS}
)

print(f"\nStarting Bayesian Optimization (Optuna, {N_BAYES_TRIALS} trials)...")
start_time = time.time()

trial_scores = []

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 10, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
    }

    model = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_split=params["min_samples_split"],
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=mae_scorer,   # negative MAE
        n_jobs=-1
    )
    mean_score = scores.mean()  # still negative MAE

    # store for logging after optimization
    trial_scores.append(mean_score)

    return mean_score  # Optuna will minimize this

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_BAYES_TRIALS, show_progress_bar=True)

bayes_time = time.time() - start_time

# Convert to positive MAE and compute best-so-far curve
mae_values = -np.array(trial_scores)               # positive MAE per trial
best_so_far = np.minimum.accumulate(mae_values)    # best (lowest) MAE so far

# Per‑trial logging (similar to your Grid code)
for i, (curr_mae, best_mae) in enumerate(zip(mae_values, best_so_far), start=1):
    trackio.log({
        "trial": int(i),
        "cv_mae": float(curr_mae),
        "best_cv_mae_so_far": float(best_mae)
    })

# Final summary logging
best_cv_mae = -study.best_value  # convert negative to positive MAE

trackio.log({
    "best_cv_mae": float(best_cv_mae),
    "time_taken": float(bayes_time),
    "n_iterations": int(N_BAYES_TRIALS),
    "best_params": study.best_params
})
trackio.finish()

print(f" Bayesian Optimization finished: {N_BAYES_TRIALS} trials in {bayes_time:.2f}s")
print(f"   Best CV MAE: {best_cv_mae:.4f}")
print(f"   Best params: {study.best_params}")


[I 2026-04-14 22:47:36,979] A new study created in memory with name: no-name-05292fc2-4cf3-4f4d-8593-7668533df7ab


* Created new run: bayesian_optuna

Starting Bayesian Optimization (Optuna, 60 trials)...


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-04-14 22:47:59,865] Trial 0 finished with value: -13851.035543271322 and parameters: {'n_estimators': 87, 'max_depth': 29, 'min_samples_split': 8}. Best is trial 0 with value: -13851.035543271322.
[I 2026-04-14 22:48:34,933] Trial 1 finished with value: -13437.7766508986 and parameters: {'n_estimators': 125, 'max_depth': 27, 'min_samples_split': 3}. Best is trial 0 with value: -13851.035543271322.
[I 2026-04-14 22:49:14,970] Trial 2 finished with value: -13595.384078216655 and parameters: {'n_estimators': 165, 'max_depth': 19, 'min_samples_split': 5}. Best is trial 0 with value: -13851.035543271322.
[I 2026-04-14 22:49:56,962] Trial 3 finished with value: -13963.283411786004 and parameters: {'n_estimators': 190, 'max_depth': 17, 'min_samples_split': 9}. Best is trial 3 with value: -13963.283411786004.
[I 2026-04-14 22:50:53,537] Trial 4 finished with value: -13966.03156379101 and parameters: {'n_estimators': 195, 'max_depth': 27, 'min_samples_split': 10}. Best is trial 4 with v

In [11]:
trackio.show()

* Running on public URL: https://9be755b5d31d076824.gradio.live


## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [29]:
# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.

# TODO: Generate and save optuna_hyperparameter_space.png# 3. Evaluation & Plots
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import optuna

sns.set_style("whitegrid")

# ---------- 3.1 Trials vs Error Plot (trials_vs_error.png) ----------

# Grid Search: cv_results_ returns mean_test_score = negative MAE
grid_mae = -grid_search.cv_results_["mean_test_score"]     # positive MAE
grid_best_so_far = np.minimum.accumulate(grid_mae)         # best (lowest) MAE up to each trial
grid_trials = np.arange(1, len(grid_best_so_far) + 1)

# Random Search: same pattern
random_mae = -random_search.cv_results_["mean_test_score"]  # positive MAE
random_best_so_far = np.minimum.accumulate(random_mae)
random_trials = np.arange(1, len(random_best_so_far) + 1)

# Optuna: study.trials[i].value-negative MAE
optuna_maes = [-t.value for t in study.trials]             # convert to positive MAE per trial
optuna_best_so_far = np.minimum.accumulate(optuna_maes)
optuna_trials = np.arange(1, len(optuna_best_so_far) + 1)

plt.figure(figsize=(10, 6))

plt.plot(grid_trials, grid_best_so_far, label="Grid Search", marker="o", linewidth=2, markersize=3)
plt.plot(random_trials, random_best_so_far, label="Random Search", marker="s", linewidth=2, markersize=3)
plt.plot(optuna_trials, optuna_best_so_far, label="Bayesian (Optuna)", marker="^", linewidth=2, markersize=3)

plt.xlabel("Number of Iterations / Trials")
plt.ylabel("Best Mean CV MAE so far")
plt.title("Trials vs. Best CV Error (Grid vs Random vs Bayesian)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("plots/trials_vs_error.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved plots/trials_vs_error.png")


# # 3.2 Optuna Hyperparameter Space Plot
# fig = optuna.visualization.plot_optimization_history(study)
# fig.write_html("plots/optuna_hyperparameter_space.html")
# print("Saved plots/optuna_hyperparameter_space.png")

Saved plots/trials_vs_error.png


In [31]:
import optuna.visualization.matplotlib as optuna_mpl
import matplotlib.pyplot as plt

# Use matplotlib backend instead of plotly
ax = optuna_mpl.plot_optimization_history(study)
plt.title("Optuna Optimization History")
plt.tight_layout()
plt.savefig("plots/optuna_hyperparameter_space.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved plots/optuna_hyperparameter_space.png")

/tmp/ipykernel_4871/942089715.py:5: ExperimentalWarning:

optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.



Saved plots/optuna_hyperparameter_space.png


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [35]:
# TODO: Print the best hyperparameters found by all 3 methods

# TODO: Train the best model found on the full X_train

# TODO: Evaluate the model on X_test (Report MAE)

# TODO: Save best_model.pkl and any necessary encoders to the models/ folder
# 4. Final Testing & Model Saving
# For comparison, convert everything to positive MAE
methods = [
    ("Grid Search", -grid_search.best_score_, grid_search.best_params_),         # positive MAE
    ("Random Search", -random_search.best_score_, random_search.best_params_),
    ("Bayesian Optimization (Optuna)", -study.best_value, study.best_params),   # best_value is negative MAE
]

print("\nBest CV MAE for each method (lower is better):")
for name, mae, params in methods:
    print(f"  {name:30s} → CV MAE = {mae:.4f} | params = {params}")

# Determine overall best method (lowest MAE)
best_method_name, best_mae, best_params = sorted(methods, key=lambda x: x[1])[0]
print(f"\nOverall best method: {best_method_name} (CV MAE = {best_mae:.4f})")

# Retrain best model on full train data
best_model = RandomForestRegressor(**best_params, random_state=42)
best_model.fit(X_train, y_train)

# Evaluate on test set
y_pred_test = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred_test)
print(f"Test MAE on test.csv: {test_mae:.4f}")

# Save final model and (optionally) best params
with open("models/best_rf_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("models/best_params.pkl", "wb") as f:
    pickle.dump(best_params, f)

print("Saved models/best_rf_model.pkl and models/best_params.pkl")


Best CV MAE for each method (lower is better):
  Grid Search                    → CV MAE = 13270.7582 | params = {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
  Random Search                  → CV MAE = 13309.7826 | params = {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}
  Bayesian Optimization (Optuna) → CV MAE = 15468.1777 | params = {'n_estimators': 128, 'max_depth': 10, 'min_samples_split': 9}

Overall best method: Grid Search (CV MAE = 13270.7582)
Test MAE on test.csv: 12422.4285
Saved models/best_rf_model.pkl and models/best_params.pkl
